# Definición
Se encarga de predecir la siguiente palabra basandose unicamente en la palabra anterior
## Formula Matemática
$$P(w_1,....w_n)\approx P(w_1)\prod_{i=2}^n P(w_i|w_{i-1})$$

In [1]:
from collections import defaultdict, Counter
import math


class ModeloBigramas:
    def __init__(self):

        self.bigramas = defaultdict(Counter)  # C(w_{i-1}, w_i)

        self.unigramas = Counter()  # C(w)

        self.vocab = set()

    def entrenar(self, oraciones):
        """oraciones: lista de listas de tokens"""

        for tokens in oraciones:
            tokens = ["<s>"] + tokens + ["</s>"]

            for i in range(len(tokens) - 1):
                w_prev, w_curr = tokens[i], tokens[i + 1]

                self.bigramas[w_prev][w_curr] += 1

                self.unigramas[w_prev] += 1

                self.vocab.add(w_prev)

                self.vocab.add(w_curr)

    def prob_bigrama(self, w_prev, w_curr, laplace=True):
        """P(w_curr | w_prev) con suavizado de Laplace"""

        V = len(self.vocab)

        num = self.bigramas[w_prev][w_curr] + (1 if laplace else 0)

        den = self.unigramas[w_prev] + (V if laplace else 0)

        return num / den if den > 0 else 1 / V

    def log_prob_oracion(self, tokens):
        """Log-probabilidad total usando log para evitar underflow"""

        tokens = ["<s>"] + tokens + ["</s>"]

        log_prob = 0.0

        for i in range(1, len(tokens)):
            p = self.prob_bigrama(tokens[i - 1], tokens[i])

            log_prob += math.log(p)

        return log_prob

    def generar(self, inicio="<s>", max_len=15):
        """Genera texto token a token siguiendo las probabilidades"""

        import random

        tokens = [inicio]

        while tokens[-1] != "</s>" and len(tokens) < max_len:
            prev = tokens[-1]

            siguientes = self.bigramas[prev]

            if not siguientes:
                break

            total = sum(siguientes.values())

            rand = random.random() * total

            acum = 0

            for word, cnt in siguientes.items():
                acum += cnt

                if rand <= acum:
                    tokens.append(word)

                    break

        return " ".join(tokens[1:-1])

In [3]:
modelo = ModeloBigramas()

# Datos de ejemplo
oraciones = [
    ["yo", "como", "arepa"],
    ["yo", "como", "arroz"],
    ["yo", "quiero", "arepa"],
    ["yo", "quiero", "hamburguesa"],
    ["yo", "quiero", "jager"],
    ["yo", "como", "perro"],
]

# Entrenar
modelo.entrenar(oraciones)

# Probar probabilidades
print("P(arepa | como):", modelo.prob_bigrama("como", "arepa"))
print("P(arroz | como):", modelo.prob_bigrama("como", "arroz"))

# Evaluar una oración
print("Log-prob:", modelo.log_prob_oracion(["yo", "como", "arepa"]))

# Generar texto
print("Generado:", modelo.generar())

P(arepa | como): 0.15384615384615385
P(arroz | como): 0.15384615384615385
Log-prob: -5.471069472325841
Generado: yo como arepa


## Suavizado de Laplace (Add-1 Smoothing)
$$\hat P(x_i)= \frac{count(x_i)+1}{N+|V|}$$